# max-back-tied-half composite — cx28: maximum_back with 50/50 tie-splitting, argnum 0 vs 1 mirror

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `max-back-tied-half`, `arg-position-back-functions`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "max-back-tied-half"
DD_ATOM_IDS = ["max-back-tied-half", "arg-position-back-functions"]
DD_SUBTOPICS = ["Backprop: max_back with tied half-mass", "Backprop: Arg-position back funcs"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`maximum(x, y)` is elementwise — the gradient lands on whichever input was larger at each position. At ties (x == y) the convention is to split the gradient mass 50/50 so total mass is conserved. Like matmul, maximum needs TWO back-fns: argnum 0 gives `dL/dx`, argnum 1 gives `dL/dy`. The masks mirror: `(x > y) + 0.5 * (x == y)` for arg-0; `(x < y) + 0.5 * (x == y)` for arg-1. They must sum to 1 everywhere — that's the mass-conservation invariant.

### Composite Exercise — maximum_back with 50/50 tie-splitting, argnum 0 vs 1 mirror

**Atoms exercised together**: `max-back-tied-half`, `arg-position-back-functions`

Implement `cx28_maximum_back(grad_out, out, x, y, argnum)` that returns:

- if `argnum == 0`: `grad_out * ((x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype))`
- if `argnum == 1`: `grad_out * ((x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype))`
- otherwise: raise `ValueError`

Verify that the masks for argnum 0 and 1 sum to `ones_like(grad_out)` at every position — that's the mass-conservation check that catches a missing 0.5 factor.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx28_maximum_back(grad_out, out, x, y, argnum):
    raise NotImplementedError

def _test_cx28():
    # Construct overlap + ties.
    x = t.tensor([1.0, 5.0, 3.0, 7.0, 7.0])
    y = t.tensor([4.0, 2.0, 3.0, 7.0, 6.0])
    out = t.maximum(x, y)
    grad_out = t.tensor([1.0, 1.0, 1.0, 1.0, 1.0])

    gx = cx28_maximum_back(grad_out, out, x, y, argnum=0)
    gy = cx28_maximum_back(grad_out, out, x, y, argnum=1)

    # Position-by-position: x<y, x>y, tie, tie, x>y.
    expected_gx = t.tensor([0.0, 1.0, 0.5, 0.5, 1.0])
    expected_gy = t.tensor([1.0, 0.0, 0.5, 0.5, 0.0])
    assert t.allclose(gx, expected_gx), f'gx wrong: got {gx}, expected {expected_gx}'
    assert t.allclose(gy, expected_gy), f'gy wrong: got {gy}, expected {expected_gy}'

    # Mass conservation: gx + gy == grad_out everywhere.
    assert t.allclose(gx + gy, grad_out), f'masses do not sum to grad_out: {gx + gy}'

    # argnum dispatch — bad argnum must raise ValueError.
    raised = False
    try: cx28_maximum_back(grad_out, out, x, y, argnum=2)
    except ValueError: raised = True
    assert raised, 'bad argnum should raise ValueError'

    # Cross-check with autograd at a non-tied case.
    xa = t.tensor([1.0, 5.0, 8.0]).requires_grad_(True)
    ya = t.tensor([4.0, 2.0, 3.0]).requires_grad_(True)
    t.maximum(xa, ya).sum().backward()
    g0 = cx28_maximum_back(t.ones(3), None, xa.detach(), ya.detach(), argnum=0)
    g1 = cx28_maximum_back(t.ones(3), None, xa.detach(), ya.detach(), argnum=1)
    assert t.allclose(g0, xa.grad), f'argnum=0 vs autograd: {g0} vs {xa.grad}'
    assert t.allclose(g1, ya.grad), f'argnum=1 vs autograd: {g1} vs {ya.grad}'
    _dd_passed.add('cx28')

_test_cx28()

<details><summary>Show solution — cx28</summary>

```python
def cx28_maximum_back(grad_out, out, x, y, argnum):
    if argnum == 0:
        # x wins where x > y; half-share at ties so mass is conserved.
        mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
        return grad_out * mask
    if argnum == 1:
        # mirror split: y wins where x < y; same half-share at ties.
        mask = (x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
        return grad_out * mask
    raise ValueError(f'maximum has args (x, y); argnum must be 0 or 1, got {argnum}')
```

Mass conservation is the smoking-gun test: `gx + gy == grad_out`. If you skipped the 0.5 at ties (using strict `<` and `>` for both), tied positions would receive zero gradient and you'd lose mass. If you used non-strict `<=` and `>=`, tied positions would receive double mass. The 50/50 split is the unique convention that preserves the total.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx28'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx28',
        'subtopics': ["Backprop: max_back with tied half-mass", "Backprop: Arg-position back funcs"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()